# Requirements: environment setup


In [ ]:
# pip install openai pandas openpyxl numpy tzlocal tqdm
# create .env file, which includes OPENAI_API_KEY and DEEPSEEK_API_KEY, in the project file folder

# Load Package

In [ ]:
import os
import time
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime
import tzlocal
import json
from tqdm import tqdm

# Loads environment variables from .env file
# You have .env file, which includes OPENAI_API_KEY and DEEPSEEK_API_KEY, in the project file folder
load_dotenv()

True

# Load Chinese Firm Name

In [ ]:
filename = "./test_firms.xlsx"

df = pd.read_excel(filename)
company_names_zh = df.iloc[:, 0].dropna().tolist()

# Autoset Parameters

In [ ]:
researcher_location = tzlocal.get_localzone()
date = datetime.now(researcher_location).strftime("%Y-%m-%d %Z %z")
runtime = datetime.now(researcher_location).strftime("%H:%M:%S")

# Cache Files

- Store raw LLM responses and avoid redundant queries

In [ ]:
CACHE_FILE = "translated_firm_data_llm.xlsx"

# Prompt

### Translate frim's name from English to Chinese

In [ ]:
def prompt_translate(name, language):
    if language == "en":
        return f"""
        Translate the following Chinese company name into accurate and standard English
        (as used in international corporate directories): {name}

        Output strictly as a valid JSON object with the following keys and values.
        Use exactly these keys. Do not add any extra fields or commentary.

        {{
            "English Name": "Answer",
        }}
        """
    elif language == "zh":
        return f"""
        请将下列中文公司名称，准确且标准地翻译为英文（参照国际企业名录中的通用格式）：{name}

        请严格遵守以下规则：
        1. 虽然本指令与范例格式使用中文撰写，但你的回复必须完全以 **JSON** 输出。
        2. 回复必须是有效的 JSON 对象，不得包含任何额外文字或解释。

        回复必须是有效的 JSON 对象，不得包含任何额外文字或解释。
        {{
            "公司英文名称": "答案"
        }}
        """

### Search for firm's information

- System Prompt: defines how the assistant should behave — it sets the tone, rules, and constraints for all responses. It's like setting the "personality" and instructions behind the scenes.

In [ ]:
def prompt_system(language):
    if language == "en":
        return f"""
        You are a knowledgeable business information assistant.
        Answer all questions based on your internal knowledge up to June 2025.

        When providing Chinese company information, reference data typically available from:
        - Qichacha (企查查): https://www.qcc.com/
        - Tianyancha (天眼查): https://www.tianyancha.com/
        - Aiqicha (爱企查): https://aiqicha.baidu.com/
        - National Enterprise Credit Information Publicity System (国家企业信用信息公示系统): https://www.gsxt.gov.cn/

        If exact values are not available, make a reasonable inference based on:
        - Typical Chinese corporate registration structures
        - Industry patterns
        - Publicly known data from similar companies

        Important Rules:
        - All fields must contain factual information - no fabrication
        - Avoid "NA" unless absolutely no information can be inferred.
        - For unavailable information, specify reason (e.g., "NA (Not disclosed)")
        - Always output in strict JSON format with the requested keys.
        """

    elif language == "zh":
        return """
        你是一位了解商业信息的专业助手。
        请根据你截至 2025 年 6 月的内部知识回答所有问题。

        在提供中国公司的信息时，请参考以下常见数据来源：
        - 企查查（Qichacha）：https://www.qcc.com/
        - 天眼查（Tianyancha）：https://www.tianyancha.com/
        - 爱企查（Aiqicha）：https://aiqicha.baidu.com/
        - 国家企业信用信息公示系统：https://www.gsxt.gov.cn/

        如果无法获取精确数据，请根据以下内容进行合理推测：
        - 中国公司注册的典型结构
        - 行业规律
        - 类似公司的公开信息

        注意事项：
        - 所有字段必须真实存在，禁止虚构。
        - 除非完全无法推断信息，否则请避免使用“NA”作为答案。
        - 若信息不存在需注明原因（如："NA（工商未披露）"。
        - 回答必须使用严格的 JSON 格式，并且只能包含指定的键，不得添加额外内容。
        """

- User Prompt: the actual input or question given by the user — it tells the assistant what task to perform.

In [ ]:
def prompt_structured_info(name, language):
    if language == "en":
        return f"""Based on your internal knowledge as of June 2025, provide the following structured company information for:
                Company Name: {name}

                Important Rules:
                - All fields must contain factual information - no fabrication
                - Output strictly as a valid JSON object with the following keys and values.
                - Use exactly these keys. Do not add any extra fields or commentary.
                - Use exactly "NA" ONLY if the value cannot be reasonably inferred at all. But Avoid "NA" unless absolutely no information can be inferred.
                - Do not use any other placeholder like "None", "Not specified", or "Unknown".
                - For unavailable information, specify reason (e.g., "NA (Not disclosed)")

                {{
                "Chairman (Full Name)": "Full name of the company's chairman. Example: 'Li Wei'",
                "State-Owned": "Yes / No / Partial — specify % if known. Example: 'Partial — 51%'",
                "Parent Company (English name)": "Full English name of the parent company. Use the Chinese name if the English name is unavailable. Example: 'China National Petroleum Corporation.'",
                "Parent Company (Chinese name)": "Full Chinese name of the parent company. Use the English name if the Chinese name is unavailable. Example: '中石油北京天然气管道有限公司'",
                "Chairman of Parent Company": "Full name of the parent company's chairman. Example: 'Wang Yong'",
                "Headquarters Address": "Full address or at least Province-City. Example: 'Beijing, China' or '9 Dongzhimen North Street, Dongcheng District, Beijing, China'",
                "Nationality": "Country of registration. Example: 'China'",
                "Operational Status": "Currently operating? Yes or No",
                "Date of Establishment": "YYYY-MM-DD or Year if only year is known. Example: '2005-08-12' or '2005'",
                "Investment Industry": "Primary industry sector. Example: 'Natural Gas Pipeline'",
                "Registered Capital": "Company's officially registered capital (注册资本) as recorded in Chinese corporate registration systems such as Qichacha, Tianyancha, Aiqicha, or National Enterprise Credit Information Publicity System.
                                    If exact value is unknown, infer a reasonable estimate based on typical ranges for similar Chinese companies.
                                    Output in the format [Amount][RMB/USD/HKD]. Example: '50,000,000[RMB]' or '2,000,000[USD]'.",
                "Legal Representative": "Full name of the legal representative. Example: 'Zhang Wei'",
                "Investors": "List all major investors in the format: [Name]([Shareholding %])[Country][Investment Amount][Currency]; separate multiple investors with ';'",
                "Data Sources": "List of data sources used, separated by ";".
                }}
                """
    elif language == 'zh':
        return f"""
                根据你截至 2025 年 6 月的知识，请为以下公司提供结构化的公司信息：公司名称：{name}

                重要规则：
                - 所有字段必须真实存在，禁止虚构。
                - 回答必须严格以有效的 JSON 对象输出，包含以下键（key）与对应的值（value）。
                - 必须完全按照以下键名输出，不可添加额外的字段或评论。
                - 仅在完全无法合理推断时，才可以使用“NA”，但除非确实没有任何可推断的信息，应尽量避免使用“NA”。
                - 不得使用 “None”、“Not specified”、“Unknown” 等其他占位词。
                - 若信息不存在需注明原因（如："NA（工商未披露）"。

                {{
                    "董事长 (全名)": "公司董事长（全名）。例如：'刘扬伟'",
                    "是否国有": "是否国有企业。填写 国资/民营/混合 - 注明百分比，如为部分国有请说明比例。例如：'混合 — 51%'",
                    "母公司 (英文名字)": "母公司（英文全称），如果没有就用中文名字。例如：'China National Petroleum Corporation.'",
                    "母公司 (中文名字)": "母公司（中文全称），如果没有就用英文名字。例如：'中石油北京天然气管道有限公司'",
                    "母公司董事长": "母公司董事长（全名）。例如：'刘扬伟'",
                    "总部地址": "总部地址，可为完整地址或至少省市。例如：'北京, 中国' 或 '北京市顺义区双河大街99号'",
                    "注册国": "公司注册所在国家。例如：'中国'",
                    "营运状态": "公司目前是否在营运。填写 是 或 否",
                    "成立日期": "公司成立日期，格式为 YYYY-MM-DD，若仅知道年份可填年份。例如：'2005-08-12' 或 '2005'",
                    "主要投资产业": "主要投资或经营行业。例如：'电子制造服务'",
                    "注册资本": "公司在中国工商注册系统（如企查查、天眼查、爱企查、国家企业信用信息公示系统）中登记的注册资本。
                                若确切数据未知，请根据类似企业的典型范围合理推测。
                                输出格式为 [金额][RMB/USD/HKD]。例如：'50,000,000[RMB]' 或 '2,000,000[USD]'。",
                    "法定代表人 (全名)": "公司法定代表人（全名）。例如：'陈巍'",
                    "投资者": "列出所有主要投资者，格式为：[名称]([持股%])[国家][投资金额][币种]；多个投资者请用 ';' 分隔",
                    "资料来源": "使用的资料来源列表，请以「；」分隔。
                }}
                """
    #"資料來源": "使用的資料來源列表，請以「；」分隔。

In [ ]:
# If cache file exists, load it into a dictionary to avoid redundant queries
def load_cache_file():
    if os.path.exists(CACHE_FILE):
        df_cache = pd.read_excel(CACHE_FILE)
        cache_dict = {}
        for _, row in df_cache.iterrows():
            # Use (Model, Date, Location, Language, Chinese Name) as the cache key
            key_cn = (row["Model"], row["Date"], row["Researcher Location"], row["Language"], row["Chinese Name"])
            cache_dict[key_cn] = (row["English Name"], row["ChatGPT Response"])
    else:
        # Initialize empty cache if file doesn't exist
        cache_dict = {}
    return cache_dict

# Query LLM

### Function: Translate frim's name from English to Chinese

In [ ]:
# Sends a prompt to the LLM to translate a Chinese company name into English
def translate_name(client, model,chinese_name, prompt):
    try:
        # Call the chat completion API with the given model and prompt, expecting JSON output
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0,
        )
        # Return the stripped JSON string from the response
        return response.choices[0].message.content.strip()
    except Exception as e:
        # Print error and return fallback string if the call fails
        print(f"Translation failed for '{chinese_name}': {e}")
        return "TRANSLATION_ERROR"

### Function Search for firm's information

In [ ]:
# Sends a structured query to the LLM (e.g., ChatGPT or DeepSeek) with system and user prompts
def query_gpt(client, model, company_name_en, prompt, system_prompt):
    try:
        # Call the chat completion API with both system and user prompts, expecting JSON output
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )

        # Extract response content and token usage
        raw_output = response.choices[0].message.content.strip()
        usage = response.usage

        # Extract token statistics, including cached tokens if available
        token_info = {
            "uploaded_tokens": usage.prompt_tokens,
            "downloaded_tokens": usage.completion_tokens,
            "cached_tokens": getattr(usage.prompt_tokens_details, "cached_tokens", 0),
            "total_tokens": usage.total_tokens
        }

        return raw_output, token_info
    except Exception as e:
        # Handle query failure by logging and returning error defaults
        print(f"GPT query failed for '{company_name_en}': {e}")
        return "QUERY_ERROR", {"uploaded_tokens": 0, "downloaded_tokens": 0, "cached_tokens": 0, "total_tokens": 0}

### Function LLM Querying

In [ ]:
# Recursively queries a company and its parent companies using LLM, with caching and translation
def query_with_parent_chain(client, model, language ,name_zh, cache_dict, prompt_translate, prompt_structured_info, prompt_system, translate_fn, query_fn, sleep_time=5, visited=None):
    # Track visited company names to avoid cycles.
    # For example, some companies list themselves as their own parent company instead of showing "NA" — in such cases, you should stop the parent query.
    if visited is None:
        visited = set()

    # Store final result rows
    results = []

    # Define recursive logic to query a company and its parent chain
    def recursive_query(name_zh_local, name_en_local=None, is_parent=False):
        # Stop if this company has already been queried
        if (name_zh_local in visited) or (name_en_local in visited):
            return

        # Construct cache key based on 5 attributes
        cache_key = (model, date, researcher_location.key, language, name_zh_local)
        used_cache = False

        # Check cache to skip redundant queries
        if cache_key in cache_dict:
            used_cache = True
            name_en_local, response = cache_dict[cache_key]
            print(f"\nFound cached result for {name_zh_local} with ({model}, {date}, {researcher_location}, {language})")
        else:
            # Step 1: Translate Chinese name to English if not already known
            if is_parent:
                print(f"\nUsing Parent Company: {name_en_local} {name_zh_local}")
            else:
                print(f"\nTranslating: {name_zh_local}")
                raw_response = translate_fn(client,model, name_zh_local,  prompt_translate(name_zh_local, language))
                response = json.loads(raw_response)
                if language == "en":
                    name_en_local = response["English Name"]
                elif language == "zh":
                    name_en_local = response["公司英文名称"]
                print(f"→ {name_en_local}")

            # Step 2: If translation fails, skip querying
            if name_en_local == "TRANSLATION_ERROR":
                response = "QUERY_SKIPPED"
                duration = 0.0
                query_start_time = query_end_time = "NA"
                token_info = {
                    "uploaded_tokens": 0,
                    "cached_tokens": 0,
                    "downloaded_tokens": 0
                }
            else:
                # Step 3: Query company info from LLM
                # If language is English, use the English company name to generate the prompt; otherwise, use the Chinese name
                prompt_query = prompt_structured_info(name_en_local if language == "en" else name_zh_local, language)
                system_prompt = prompt_system(language)
                print(f"Querying GPT for: {name_en_local if language == 'en' else name_zh_local}")

                # Measure query start time
                start_time = time.time()
                query_start_time = datetime.now().strftime("%H:%M:%S")

                # Send query to the LLM
                raw_response, token_info = query_fn( client, model, name_en_local, prompt_query, system_prompt)

                # Measure query end time
                query_end_time = datetime.now().strftime("%H:%M:%S")
                end_time = time.time()

                # Total query duration in seconds
                duration = round(end_time - start_time, 3)

                response = json.loads(raw_response)

                # Store result in cache
                cache_dict[cache_key] = (name_en_local, response)

        # Mark current company as visited
        visited.add(name_zh_local)
        visited.add(name_en_local)

        # If not from cache, append to results
        if not used_cache:
            results.append([
                name_zh_local, # Chinese company name
                name_en_local, # English company name
                response, # Raw JSON response
                query_start_time, # Start time
                query_end_time, # End time
                duration, # Query duration (sec)
                1 if is_parent else 0, # Is parent company?
                token_info["uploaded_tokens"],
                token_info["cached_tokens"],
                token_info["downloaded_tokens"],
            ])

            # Step 4: Recursively query parent company if available
            try:
                if language == "en":
                    parent_company_en = response["Parent Company (English name)"]
                    parent_company_zh = response["Parent Company (Chinese name)"]
                elif language == "zh":
                    parent_company_en = response["母公司 (英文名字)"]
                    parent_company_zh = response["母公司 (中文名字)"]

                # If parent company exists (not starting with "NA"), recursively query it
                if not parent_company_zh.startswith("NA") or not parent_company_en.startswith("NA"):
                    print(f"Found Parent Company: {parent_company_en} {parent_company_zh}")
                    recursive_query(name_zh_local=parent_company_zh, name_en_local=parent_company_en, is_parent=True)
            except Exception as e:
                print(f"Could not extract parent company for {name_en_local}: {e}")

    # Begin recursive query
    recursive_query(name_zh)

    return results

### Function Save to Cache

In [ ]:
def save_to_cache_file(model, language, all_results):
# Create an array to store fixed query parameters for each result (model, date, time, location, language)
    param_ary = np.empty((np.array(all_results).shape[0], 5), dtype=object)
    param_ary[:, 0] = model # Model used (e.g., gpt-4o)
    param_ary[:, 1] = date # Query date
    param_ary[:, 2] = runtime # Query time
    param_ary[:, 3] = researcher_location.key # Location of researcher
    param_ary[:, 4] = language # Language used for prompt
    # Save output results to cache file if there are any new results

    if len(all_results) != 0:
        # Combine fixed parameters with query results into a DataFrame
        df_out = pd.DataFrame(np.hstack((param_ary, all_results)), columns=["Model", "Date", "Runtime", "Researcher Location", "Language",
                                                                            "Chinese Name", "English Name", "ChatGPT Response",
                                                                            "Query Start Time", "Query End Time", "Query Duration (sec)",
                                                                            "Is Parent",
                                                                            "Uploaded Tokens", "Cached Tokens", "Downloaded Tokens"])
        # Append to existing cache file if it exists, otherwise create new file
        if os.path.exists(CACHE_FILE):
            df_cache = pd.read_excel(CACHE_FILE)
            df_combined = pd.concat([df_cache, df_out], ignore_index=True)
            df_combined.to_excel(CACHE_FILE, index=False)
        else:
            df_out.to_excel(CACHE_FILE, index=False)

    print(f"\nAll done. Cached results saved to {CACHE_FILE}")

# Check LLM Response Format

This is a validation step to ensure each company’s LLM Response matches the exact format required in the prompt. This ensures that even if some companies have incorrectly formatted LLM Responses, the script will still generate the final output file containing all valid company data. Problematic entries just won’t block the process:
- If the format is incorrect, in the final output file, the Chinese Name and English Name will be labeled “(format incorrect),” all other fields are set to NA
- The original entry is logged in invalid_llm_responses.xlsx along with the error message for later manual review

In [ ]:
"""
This dictionary represents the exact output format I require from the LLM.
It is used as a reference template to validate whether the actual LLM response matches the expected structure.
The LLM’s response must have exactly the same set of keys—no missing keys and no additional keys—regardless of the values.
"""
def output_format(language):
    """
    Return a dictionary template with the required keys for a given language,
    initializing all values to "NA".

    Parameters:
    language (str): Language code, either "en" for English keys or "zh" for Chinese keys.

    Returns:
    dict: A dictionary where each required key maps to "NA".
    """
    if language == "en":
        # Define the list of required keys for English output format
        wanted_keys_en = [
            "Chairman (Full Name)",
            "State-Owned",
            "Parent Company (English name)",
            "Parent Company (Chinese name)",
            "Chairman of Parent Company",
            "Headquarters Address",
            "Nationality",
            "Operational Status",
            "Date of Establishment",
            "Investment Industry",
            "Registered Capital",
            "Legal Representative",
            "Investors",
            "Data Sources"
        ]
        # Create a dictionary with all keys initialized to "NA"
        return {k: "NA" for k in wanted_keys_en}
    elif language == 'zh':
        # Define the list of required keys for Chinese output format
        wanted_keys_zh = [
            "董事长 (全名)",
            "是否国有",
            "母公司 (英文名字)",
            "母公司 (中文名字)",
            "母公司董事长",
            "总部地址",
            "注册国",
            "营运状态",
            "成立日期",
            "主要投资产业",
            "注册资本",
            "法定代表人 (全名)",
            "投资者",
            "资料来源"
        ]
        # Create a dictionary with all keys initialized to "NA"
        return {k: "NA" for k in wanted_keys_zh}
    #"資料來源"

In [ ]:
def validate_llm_response(llm_response_dict, output_format_dict):
    # Check if the LLM response is a dictionary
    if not isinstance(llm_response_dict, dict):
        return False, "Not a dict type"

    # Get the set of keys from the LLM response
    llm_keys = set(llm_response_dict.keys())

    # Get the set of required keys from the output format dictionary
    required_keys_set = set(output_format_dict)

    # Check if the keys match exactly
    if llm_keys != required_keys_set:
        missing = required_keys_set - llm_keys
        extra = llm_keys - required_keys_set
        msg = []
        if missing:
            msg.append(f"Missing keys: {missing}") # Add missing keys message
        if extra:
            msg.append(f"Extra keys: {extra}") # Add extra keys message
        return False, "; ".join(msg) # Return error details

    # If all keys match exactly, the format is correct
    return True, "Format is correct"

In [ ]:
def process_llm_results(model, language, all_results):
    # Filter results but keep invalid entries with a note
    processed_results = [] # Store all results (valid and invalid, with modifications for invalid ones)
    invalid_entries = []  # Store only invalid entries for separate Excel output
    required_keys = output_format(language) # Get the required keys for LLM output based on language

    for entry in all_results:
        llm_response_dict = entry[2]  # Assuming index 2 holds the LLM response dictionary
        is_valid, msg = validate_llm_response(llm_response_dict, required_keys) # Validate format

        if is_valid:
            # If valid, keep the entry unchanged
            processed_results.append(entry)
        else:
            # If invalid, print debug message
            print(f"Invalid format for {entry[0]}: {msg}")

            # Create a modified entry indicating format issue
            new_entry = list(entry)  # Copy to modify
            new_entry[0] = f"(format incorrect) {entry[0]}" # Mark Chinese Name as incorrect
            new_entry[1] = f"(format incorrect) {entry[1]}" # Mark English Name as incorrect
            new_entry[2] = required_keys # Replace with default required format but the value of each key shows NA
            processed_results.append(new_entry)  # Append modified invalid entry

            # Save details of the invalid entry for Excel export
            invalid_entries.append({
                "Model": model,
                "Language": language,
                "Location": researcher_location,
                "Time": date + " " + runtime,
                "Chinese Name": entry[0], # Original Chinese Name (unchanged)
                "English Name": entry[1], # Original English Name (unchanged)
                "LLM Response": entry[2], # Original LLM Response
                "Message": msg # Error message from validation
            })

    # Replace all_results with the processed list (containing both valid and modified invalid entries)
    all_results = processed_results

    # Save invalid entries into a separate Excel file for manual review (append mode)
    if invalid_entries:
        invalid_path = "invalid_llm_responses.xlsx"
        df_invalid_new = pd.DataFrame(invalid_entries)

        if os.path.exists(invalid_path):
            # Read the old file and merge with new data
            df_invalid_old = pd.read_excel(invalid_path)
            df_invalid_all = pd.concat([df_invalid_old, df_invalid_new], ignore_index=True)
        else:
            # If no old file exists, just use the new data
            df_invalid_all = df_invalid_new

        df_invalid_all.to_excel(invalid_path, index=False)
        print(f"Invalid entries appended to {invalid_path}")
    else:
        print("No invalid entries found.")
    return all_results

# Manage output

In [ ]:
# Convert raw result list into a structured numpy array for final output
def process_output(model, language, all_results_ls):

    output_ls = []

    for i in range(len(all_results_ls)):

        # JSON response
        response = all_results_ls[i][2]

        # Chinese Firm Name
        name_zh = all_results_ls[i][0]

        # English firm name
        name_en = all_results_ls[i][1]

        if language == "en":
            # Chairman
            chairman = response["Chairman (Full Name)"]

            # State-Owned
            soe = response["State-Owned"]

            # Parent Company
            parent_company_zh = response["Parent Company (Chinese name)"]
            parent_company_en = response["Parent Company (English name)"]

            # Chairman of Parent Company
            chairman_parent = response["Chairman of Parent Company"]

            # Headquarters Address
            hq_address = response["Headquarters Address"]

            # Nationality
            nationality = response["Nationality"]

            # Operational Status
            operational_status = response["Operational Status"]

            # Date of Establishment
            establishment_date = response["Date of Establishment"]

            # Investment Industry
            investment_industry = response["Investment Industry"]

            # Registered Capital
            registered_capital = response["Registered Capital"]

            # Legal Representative
            legal_rep = response["Legal Representative"]

            # Investor list: company/share/country/amount
            investors = response["Investors"]

            # Data Sources
            sources = response["Data Sources"]

        elif language == "zh":
            # Chairman
            chairman = response["董事长 (全名)"]

            # State-Owned
            soe = response["是否国有"]

            # Parent Company
            parent_company_zh = response["母公司 (中文名字)"]
            parent_company_en = response["母公司 (英文名字)"]

            # Chairman of Parent Company
            chairman_parent = response["母公司董事长"]

            # Headquarters Address
            hq_address = response["总部地址"]

            # Nationality
            nationality = response["注册国"]

            # Operational Status
            operational_status = response["营运状态"]

            # Date of Establishment
            establishment_date = response["成立日期"]

            # Investment Industry
            investment_industry = response["主要投资产业"]

            # Registered Capital
            registered_capital = response["注册资本"]

            # Legal Representative
            legal_rep = response["法定代表人 (全名)"]

            # Investor list: company/share/country/amount
            investors = response["投资者"]

            # Data Sources
            sources = response["资料来源"]

        # Other metadata
        query_start_time = all_results_ls[i][3]
        query_end_time = all_results_ls[i][4]
        query_duration = all_results_ls[i][5]
        is_parent = all_results_ls[i][6]
        uploaded_tokens = all_results_ls[i][7]
        cached_tokens = all_results_ls[i][8]
        downloaded_tokens = all_results_ls[i][9]

        # Combine all values into a row
        output_ls.append([query_start_time, query_end_time, query_duration, uploaded_tokens, cached_tokens, downloaded_tokens,
                          is_parent, name_zh, name_en, chairman, soe, parent_company_zh, parent_company_en, chairman_parent,
                          hq_address, nationality, operational_status, establishment_date,
                          investment_industry, registered_capital, legal_rep, investors, sources])

    # Convert to numpy array
    output2_ary = np.array(output_ls)

    # Create fixed parameter array (model, date, time, location, language)
    output1_ary = np.empty((output2_ary.shape[0], 5), dtype=object)
    output1_ary[:, 0] = model
    output1_ary[:, 1] = date
    output1_ary[:, 2] = runtime
    output1_ary[:, 3] = researcher_location.key
    output1_ary[:, 4] = language

    # Combine fixed parameters and extracted response values
    return np.hstack((output1_ary, output2_ary))

In [ ]:
'''
Compute completeness score for a row:
0 = all fields start with "NA" (missing)
1 = some fields start with "NA" (partially complete)
2 = no fields start with "NA" (fully complete)
'''
def completeness_score(row):
    na_mask = row.astype(str).str.startswith("NA")

    if na_mask.all():
        return 0
    elif na_mask.any():
        return 1
    else:
        return 2

In [ ]:
def output_result_to_file(model, language, all_results):
    # If there are results, process and save them to output.xlsx
    if len(all_results) != 0:
        # Convert results to structured array and define output column names
        ary_output = process_output(model, language, all_results)
        columns_output = [
            "Model",
            "Date",
            "Runtime",
            "Researcher Location",
            "Language",
            "Query Start Time",
            "Query End Time",
            "Query Duration (sec)",
            "Uploaded Tokens",
            "Cached Tokens",
            "Downloaded Tokens",
            "Is Parent",
            "Chinese Firm Name",
            "English firm name",
            "Chairman",
            "SOE",
            "Parent Company (Chinese)",
            "Parent Company (English)",
            "Chairman Parent Company",
            "Headquarters Address",
            "Nationality",
            "Status",
            "Establishment year",
            "Investment industry",
            "Registered Capital",
            "Legal Representative",
            "Investor list: company/share/country/amount",
            "Data Sources"
        ]


        # Create DataFrame from result array
        df_output = pd.DataFrame(ary_output, columns=columns_output)

        # Add completeness score based on whether fields start with "NA"
        df_output["Result Completeness"] = df_output.iloc[:, -11:].apply(completeness_score, axis=1)

        # Move "Result Completeness" column just after the name of firm column
        cols = list(df_output.columns)
        cols.insert(-11, cols.pop(cols.index("Result Completeness")))
        df_output = df_output[cols]

        # Combine the new results with the old output.xlsx; otherwise, create new file
        if os.path.exists("output.xlsx"):
            df_cache_output = pd.read_excel("output.xlsx")

            # Combine (append) the new results with the existing ones
            df_combined = pd.concat([df_cache_output, df_output], ignore_index=True)

            # Overwrite the file with the combined results
            df_combined.to_excel("output.xlsx", index=False)
        else:
            df_output.to_excel("output.xlsx", index=False)
        print("All done. The query output is organized and saved to output.xlsx.")
    else:
        # If no new results, skip saving and inform user
        print(f"Queries with ({model}, {date}, {researcher_location}, {language}) have already been saved to output.xlsx.")
        pass


# Run Pipeline

In [ ]:
param_sets = [
    {
       'LLM' : "ChatGPT",
       'model' : "gpt-4.1",
       'language' : "en"
    },
    {
       'LLM' : "ChatGPT",
       'model' : "gpt-4.1",
       'language' : "zh"
    },
    {
        'LLM' : "DeepSeek",
        'model' : "deepseek-chat",
        'language' : "en"
    },
    {
        'LLM' : "DeepSeek",
        'model' : "deepseek-chat",
        'language' : "zh"
    }
]


for param in param_sets:
    LLM = param.get('LLM')
    model = param.get('model')
    language = param.get('language')
    print(LLM, model, language)




    if LLM == "ChatGPT":
        client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY", "KEY")
        )
    elif LLM == "DeepSeek":
        client = OpenAI(
            api_key=  os.getenv("DEEPSEEK_API_KEY", "KEY"),
            base_url= "https://api.deepseek.com",
        )

    cache_dict = load_cache_file()

    # Loop through all Chinese company names and query each one (including its parent chain), appending results
    all_results = []
    for company in tqdm(company_names_zh, desc="Processing companies"):
        results = query_with_parent_chain(
            client = client,
            model = model,
            language=language,
            name_zh=company,
            cache_dict=cache_dict,
            prompt_translate=prompt_translate,
            prompt_structured_info=prompt_structured_info,
            prompt_system=prompt_system,
            translate_fn=translate_name,
            query_fn=query_gpt,
            sleep_time=5
        )
        all_results.extend(results)

    save_to_cache_file(model, language, all_results)
    processed_results = process_llm_results(model, language, all_results)
    output_result_to_file(model, language, processed_results)


ChatGPT gpt-4.1 en


Processing companies: 100%|██████████| 99/99 [00:00<00:00, 10196.85it/s]


Found cached result for 樟树市马天尼文化创意有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 赣州市宝盛玻璃制品有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 喀什尚金品珠宝有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 福州仓山维利贸易有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 黑龙江飞鹤乳业有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 常州璟德弘投资有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 深圳市臻富材料科学有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 乌鲁木齐爱邻友企业管理有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 英皇电影城（安徽）有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles, en)

Found cached result for 烟台赤外线桑拿制品有限公司 with (gpt-4.1, 2025-08-23 PDT -0700, America/Los_Angeles